In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import pysam
import glob
import os

import statistics
from scipy.stats import beta, chi2_contingency
from scipy.optimize import nnls
from scipy.stats import rankdata
from scipy.special import ndtri
from scipy.stats import norm
from scipy.stats import chi2

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import statsmodels.stats.diagnostic as smd

from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

from inference import COSMIC, EM

# Read PC data

In [ ]:
fams_inds = {}
ind_fam = {}

fams_df = pd.read_csv('ukb_families.csv')

for i, row in fams_df.iterrows():

    inds = [int(ind) for ind in row['INDS'].split(',')]
    fam = 'ukb_'
    if i < 1002:
        fam += 'trio_'
    elif i >= 1002:
        fam += 'sib_'
    fam += f'{inds[0]}'
    for ind in inds[1:]:
        fam += f'_{ind}'
    
    fams_inds[fam] = inds
    for ind in inds:
        ind_fam[ind] = fam

In [ ]:
inds_pcs = {}

pcs_df = pd.read_csv('ukb_96_type_dnms_pc_added.csv')

for _, row in pcs_df.iterrows():
    inds_pcs[int(row['off_pc'])] = [row[f'pc{x+1}'] for x in range(6)]

In [ ]:
pcs_aou_df = pd.read_csv('aou_gt_lof_indel_snps_ages_added.txt', sep = '\t')

for _, row in pcs_aou_df.iterrows():
    fam = f"aou_{row['source']}_{row['fid']}"
    inds_pcs[fam] = [row[f'pc{x+1}'] for x in range(6)]

In [ ]:
len(inds_pcs)

# Read de novo mutation spectrum data

In [ ]:
def get_types(dnms):

    types = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                for prev in ['A', 'C', 'G', 'T']:
                    for next in ['A', 'C', 'G', 'T']:
                        types[f'{prev}[{ref}>{alt}]{next}'] = 0

    for tp, cnt in dnms.items():
        types[tp] = cnt
    
    return types

In [ ]:
fams_rates = {}

fams_c_a_rates = {}
fams_c_g_rates = {}
fams_c_t_rates = {}
fams_cpg_tpg_rates = {}

fams_t_a_rates = {}
fams_t_c_rates = {}
fams_t_g_rates = {}

fams_types = {}
fams_father_age = {}
fams_mother_age = {}
fams_source = {}

In [ ]:
fams_ukb_dnms_df = pd.read_csv('ukb_96_type_dnms.csv')

start_idx = fams_ukb_dnms_df.columns.get_loc('FAM')+1
end_idx = fams_ukb_dnms_df.columns.get_loc('rate')
cols = fams_ukb_dnms_df.columns[start_idx:end_idx]

for _, row in fams_ukb_dnms_df.iterrows():

    fam = f"ukb_{row['source']}_{row['FAM']}"
    fams_rates[fam] = row['rate']
    
    fams_c_a_rates[fam] = row['c_a_rate']
    fams_c_g_rates[fam] = row['c_g_rate']
    fams_c_t_rates[fam] = row['c_t_rate']
    fams_cpg_tpg_rates[fam] = row['cpg_tpg_rate']

    fams_t_a_rates[fam] = row['t_a_rate']
    fams_t_c_rates[fam] = row['t_c_rate']
    fams_t_g_rates[fam] = row['t_g_rate']
    
    fams_types[fam] = get_types(row[cols])
    if pd.isna(row['mean_father_age']) == False:
        fams_father_age[fam] = row['mean_father_age']
    if pd.isna(row['mean_mother_age']) == False:
        fams_mother_age[fam] = row['mean_mother_age']
    fams_source[fam] = f"ukb_{row['source']}"

In [ ]:
fams_aou_dnms_df = pd.read_csv('families_COSMIC_96_mutation_rates.txt', sep = '\t')

for _, row in fams_aou_dnms_df.iterrows():
    
    fam = f"aou_{row['source']}_{row['fid']}"
    fams_rates[fam] = row['average_mutation_rate']
    
    fams_c_a_rates[fam] = sum([float(rate) for rate in row['c_a_rates'].split('|')])/len(row['c_a_rates'].split('|'))
    fams_c_g_rates[fam] = sum([float(rate) for rate in row['c_g_rates'].split('|')])/len(row['c_g_rates'].split('|'))
    fams_c_t_rates[fam] = sum([float(rate) for rate in row['c_t_rates'].split('|')])/len(row['c_t_rates'].split('|'))
    fams_cpg_tpg_rates[fam] = sum([float(rate) for rate in row['cpg_rates'].split('|')])/len(row['cpg_rates'].split('|'))

    fams_t_a_rates[fam] = sum([float(rate) for rate in row['t_a_rates'].split('|')])/len(row['t_a_rates'].split('|'))
    fams_t_c_rates[fam] = sum([float(rate) for rate in row['t_c_rates'].split('|')])/len(row['t_c_rates'].split('|'))
    fams_t_g_rates[fam] = sum([float(rate) for rate in row['t_g_rates'].split('|')])/len(row['t_g_rates'].split('|'))

fams_aou_dnms_df = pd.read_csv('aou_gt_lof_indel_snps_ages_added.txt', sep = '\t')

start_idx = fams_aou_dnms_df.columns.get_loc('average_mutation_rate')+1
end_idx = fams_aou_dnms_df.columns.get_loc('tot_mutations')
cols = fams_aou_dnms_df.columns[start_idx:end_idx]

for _, row in fams_aou_dnms_df.iterrows():

    fam = f"aou_{row['source']}_{row['fid']}"
    fams_types[fam] = get_types(row[cols])
    if pd.isna(row['father_age']) == False:
        fams_father_age[fam] = row['father_age']
    if pd.isna(row['mother_age']) == False:
        fams_mother_age[fam] = row['mother_age']
    fams_source[fam] = f"aou_{row['source']}"

In [ ]:
mean_father_age_source = {}
mean_mother_age_source = {}

for source in ['ukb_trio', 'ukb_sib', 'aou_trio', 'aou_sib']:
    
    father_ages_source = [fams_father_age[fam] for fam in fams_father_age if fams_source[fam] == source]
    mean_father_age_source[source] = np.mean(father_ages_source)

    mother_ages_source = [fams_mother_age[fam] for fam in fams_mother_age if fams_source[fam] == source]
    mean_mother_age_source[source] = np.mean(mother_ages_source)

fams_father_age_imputed = {}
fams_mother_age_imputed = {}

for fam in fams_types:
    if fam not in fams_father_age:
        fams_father_age[fam] = mean_father_age_source[fams_source[fam]]
        fams_father_age_imputed[fam] = True
    if fam not in fams_mother_age:
        fams_mother_age[fam] = mean_mother_age_source[fams_source[fam]]
        fams_mother_age_imputed[fam] = True

In [ ]:
source_fams = {}

for fam in fams_source:
    if fams_source[fam] not in source_fams:
        source_fams[fams_source[fam]] = []
    source_fams[fams_source[fam]].append(fam)

# Read genes metadata

In [ ]:
genes = {}

# genes_df = pd.read_csv('repair_genes_map.txt', sep = ' ', header = None)
genes_df = pd.read_csv('repair_genes_map_filtered.txt', sep = ' ', header = None)

for _, row in genes_df.iterrows():
    if row[3] == 'X':
        continue
    genes[row[0]] = True

# Trios

In [ ]:
fams_lofs_genes = {fam: {gene: 0.0 for gene in genes} for fam in fams_types}

ukb_lofs_df = pd.read_csv('lofs_genes_ukb.csv')

for i, row in ukb_lofs_df.iterrows():

    fam = 'ukb_'
    if i < 1002:
        fam += 'trio_'
    elif i >= 1002:
        fam += 'sib_'
    fam += f"{row['FAM']}"
    for gene in ukb_lofs_df.columns:
        if gene in genes:
            fams_lofs_genes[fam][gene] = row[gene]

In [ ]:
fams_miss_genes = {fam: {gene: 0.0 for gene in genes} for fam in fams_types}

ukb_miss_df = pd.read_csv('miss_genes_ukb.csv')

for i, row in ukb_miss_df.iterrows():
    
    fam = 'ukb_'
    if i < 1002:
        fam += 'trio_'
    elif i >= 1002:
        fam += 'sib_'
    fam += f"{row['FAM']}"
    for gene in ukb_miss_df.columns:
        if gene in genes:
            fams_miss_genes[fam][gene] = row[gene]

In [ ]:
trios_aou_lofs_df = pd.read_csv('aou_gt_lof_indel_snps_ages_added.txt', sep = '\t')

trios_aou_lofs_df = trios_aou_lofs_df.rename(columns = {
    'C20orf196': 'SHLD1',
    'FAM35A': 'SHLD2',
    'AC008560.1': 'SHLD3',
    'H2AFX': 'H2AX'
})

start_idx = trios_aou_lofs_df.columns.get_loc('mother_age')+1
cols = trios_aou_lofs_df.columns[start_idx:]

for _, row in trios_aou_lofs_df.iterrows():
    
    fam = f"aou_{row['source']}_{row['fid']}"
    for gene in cols:
        if gene in genes:
            fams_lofs_genes[fam][gene] = row[gene]

In [ ]:
trios_aou_miss_df = pd.read_csv('aou_gt_missense_snps_ages_added.txt', sep = '\t')

trios_aou_miss_df = trios_aou_miss_df.rename(columns = {
    'C20orf196': 'SHLD1',
    'FAM35A': 'SHLD2',
    'AC008560.1': 'SHLD3',
    'H2AFX': 'H2AX'
})

start_idx = trios_aou_miss_df.columns.get_loc('mother_age')+1
cols = trios_aou_miss_df.columns[start_idx:]

for _, row in trios_aou_miss_df.iterrows():
    
    fam = f"aou_{row['source']}_{row['fid']}"
    for gene in cols:
        if gene in genes:
            fams_miss_genes[fam][gene] = row[gene]

# Model

In [ ]:
def get_carriers(thre, mutator):

    carriers = {gene: {} for gene in genes}
        
    if mutator == 'lof':
        for fam in fams_types:
            for gene in genes:
                if fams_lofs_genes[fam][gene] >= thre:
                    carriers[gene][fam] = True
    elif mutator == 'missense':
        for fam in fams_types:
            for gene in genes:
                if fams_miss_genes[fam][gene] >= thre:
                    carriers[gene][fam] = True
    else:
        for fam in fams_types:
            for gene in genes:
                if fams_lofs_genes[fam][gene]+fams_miss_genes[fam][gene] >= thre:
                    carriers[gene][fam] = True

    return carriers

In [ ]:
def merge_types(fams, types):

    types_merged = {tp: 0 for tp in types[fams[0]]}
    
    for fam in fams:
        for tp in types[fam]:
            types_merged[tp] += types[fam][tp]
            
    return types_merged

In [ ]:
def plot_qq(pvals, alpha = 0.05, ci = False):

    n = len(pvals)
    
    pvals = np.array(pvals)
    expected = -np.log10(np.linspace(1/(n+1), 1, n))
    observed = -np.log10(np.sort(pvals))

    ci_lower = -np.log10(beta.ppf(alpha/2, np.arange(1, n+1), np.arange(n, 0, -1)))
    ci_upper = -np.log10(beta.ppf(1-alpha/2, np.arange(1, n+1), np.arange(n, 0, -1)))

    if ci == True:
        plt.fill_between(expected, ci_lower, ci_upper, color = 'lightgrey', alpha = 0.5, label = '95% CI')
        plt.legend()

    plt.plot(expected, observed, 'o', markersize = 3)
    plt.plot([0, max(expected)], [0, max(expected)], 'r--')
    
    plt.xlabel('Expected -log10(p)')
    plt.ylabel('Observed -log10(p)')
    plt.grid(True)
    plt.show()
    plt.clf()

## SBS5

In [ ]:
cosmic_df = pd.read_csv('COSMIC_v3.5_SBS_GRCh38.txt', sep = '\t').set_index('Type')
cosmic_df = cosmic_df[['SBS5', 'SBS1', 'SBS39', 'SBS12', 'SBS16', 'SBS3', 'SBS6', 'SBS14', 'SBS15', 'SBS20', 'SBS21', 'SBS26', 
                       'SBS44', 'SBS10a', 'SBS10b', 'SBS10c', 'SBS10d', 'SBS18', 'SBS30', 'SBS36']]
cosmic_mutation_types = cosmic_df.index.tolist()
cosmic_X = cosmic_df.values

def get_sbs_counts(fams, sbss):

    y = merge_types(fams, fams_types)
    y = np.array([y.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
    betas, _ = nnls(cosmic_X, y)

    y_reconstructed = cosmic_X @ betas
    cos_sim = cosine_similarity(y.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0]
    
    sbss_idxs = [cosmic_df.columns.get_loc(sbs) for sbs in sbss]
    return betas[sbss_idxs].sum(), sum(betas), cos_sim
    # return betas[sbss_idxs].sum(), y.sum(), cos_sim

In [ ]:
def sbs_permutation_carrier_status(thre, mutator, genes, sbss, change, B = 1000):

    carriers = get_carriers(thre, mutator)

    gene_fracts = {}
    gene_results = []

    for _, gene in tqdm(enumerate(genes)):

        noncarriers = [fam for fam in fams_types if fam not in carriers[gene]]

        carrier_cnt, carrier_total, _ = get_sbs_counts(list(carriers[gene]), sbss)
        noncarrier_cnt, noncarrier_total, _ = get_sbs_counts(noncarriers, sbss)
        carrier_fract = carrier_cnt / carrier_total
        noncarrier_fract = noncarrier_cnt / noncarrier_total

        carriers_source_cnts = {}
        for fam in carriers[gene]:
            source = fams_source[fam]
            if source not in carriers_source_cnts:
                carriers_source_cnts[source] = 0
            carriers_source_cnts[source] += 1

        null_delta = []
        for b in range(B):
            
            carriers_b = set()
            for group in carriers_source_cnts:
                carriers_b.update(random.sample(source_fams[group], carriers_source_cnts[group]))
            
            noncarriers_b = [fam for fam in fams_types if fam not in carriers_b]

            carrier_b_cnt, carrier_b_total, _ = get_sbs_counts(list(carriers_b), sbss)
            noncarrier_b_cnt, noncarrier_b_total, _ = get_sbs_counts(noncarriers_b, sbss)
            carrier_b_fract = carrier_b_cnt / carrier_b_total
            noncarrier_b_fract = noncarrier_b_cnt / noncarrier_b_total
            
            null_delta.append(carrier_b_fract - noncarrier_b_fract)
            
        carriers_delta = carrier_fract - noncarrier_fract

        if change == 'increase':
            pval = (np.sum(null_delta >= carriers_delta)+1)/(B+1)
        elif change == 'decrease':
            pval = (np.sum(null_delta <= carriers_delta)+1)/(B+1)
        else:
            pval = (np.sum(np.abs(null_delta) >= abs(carriers_delta))+1)/(B+1)
            
            
        gene_results.append({'gene': gene, 
                             'num_carriers': len(carriers[gene]), 
                             'num_noncarriers': len(noncarriers), 
                             'carrier_fract': carrier_fract,
                             'noncarrier_fract': noncarrier_fract,
                             'delta': carrier_fract - noncarrier_fract,
                             'pval': pval})

    gene_results_df = pd.DataFrame(gene_results)
    
    print(gene_results_df[['gene', 'num_carriers', 'num_noncarriers', 'carrier_fract', 
                           'noncarrier_fract', 'delta', 'pval']].to_string())

    return gene_results_df

## Burden test

In [ ]:
def get_pcs(fam):
        
    if 'ukb' in fam:
        fam_pcs = next((inds_pcs[ind] for ind in fams_inds[fam] if ind in inds_pcs), None)
        if fam_pcs == None:
            print('No PCs', fams_inds[fam])
    elif 'aou' in fam:
        fam_pcs = inds_pcs[fam] if fam in inds_pcs else None
        if fam_pcs == None:
            print('No PCs', fam)

    return fam_pcs

In [ ]:
def genomic_control(pvals):
    
    chisq = chi2.isf(pvals, df = 1)
    lambda_gc = np.median(chisq) / chi2.median(df = 1)
    pvals_gc = chi2.sf(chisq / lambda_gc, df = 1)
    return pvals_gc, lambda_gc

In [ ]:
def get_gts(mutator):

    if mutator == 'lof':
        gts = {}
        for gene in genes:
            gts[gene] = {}
            for fam in fams_types:
                gts[gene][fam] = min(fams_lofs_genes[fam][gene], 2.0)
    elif mutator == 'missense':
        gts = {}
        for gene in genes:
            gts[gene] = {}
            for fam in fams_types:
                gts[gene][fam] = min(fams_miss_genes[fam][gene], 2.0)
    else:
        gts = {}
        for gene in genes:
            gts[gene] = {}
            for fam in fams_types:
                gts[gene][fam] = min(fams_lofs_genes[fam][gene]+fams_miss_genes[fam][gene], 2.0)

    return gts

In [ ]:
def gwas_rate_dosage(mutator, tp, recessive = False):
    
    def get_rates(gene):

        carriers = pheno_geno[pheno_geno[gene] >= 1]['Rate']
        noncarriers = pheno_geno[pheno_geno[gene] < 1]['Rate']

        return pd.Series({
            'N_carriers': int(carriers.shape[0]),
            'N_noncarriers': int(noncarriers.shape[0]),
            'mean_rate_carriers': carriers.mean(),
            'mean_rate_noncarriers': noncarriers.mean(),
        })

    gts = get_gts(mutator)
    gts_lofs = get_gts('lof')
    gts_miss = get_gts('missense')

    rows = []
    
    for fam in fams_types:
        
        fam_rate = (fams_rates[fam] if tp == 'total' else 
                    (fams_c_a_rates[fam] if tp == 'C>A' else 
                     (fams_c_g_rates[fam] if tp == 'C>G' else
                      (fams_c_t_rates[fam] if tp == 'C>T' else
                       (fams_cpg_tpg_rates[fam] if tp == 'CpG>TpG' else 
                        (fams_t_a_rates[fam] if tp == 'T>A' else
                         (fams_t_c_rates[fam] if tp == 'T>C' else
                          (fams_t_g_rates[fam] if tp == 'T>G' else 0))))))))
    
        fam_pcs = get_pcs(fam)

        fam_father_age = fams_father_age[fam]
        fam_mother_age = fams_mother_age[fam]

        fam_source = fams_source[fam]

        if recessive == True:
            fam_gts = {gene: 1 if gts[gene][fam] >= 2 else 0 for gene in gts}
            fam_gts_lofs = {f'{gene}_lofs': 1 if gts_lofs[gene][fam] >= 2 else 0 for gene in gts_lofs}
            fam_gts_miss = {f'{gene}_miss': 1 if gts_miss[gene][fam] >= 2 else 0 for gene in gts_miss}
        else:
            fam_gts = {gene: gts[gene][fam] for gene in gts}
            fam_gts_lofs = {f'{gene}_lofs': gts_lofs[gene][fam] for gene in gts_lofs}
            fam_gts_miss = {f'{gene}_miss': gts_miss[gene][fam] for gene in gts_miss}

        row = {
            'Family': fam,
            'Rate': fam_rate,
            'Total': sum(fams_types[fam].values()),
            **{f'PC{i+1}': fam_pcs[i] for i in range(6)},
            'Paternal_age': fam_father_age,
            'Maternal_age': fam_mother_age,
            'Biobank': 0 if 'ukb' in fam_source else 1,
            'Source': 0 if 'trio' in fam_source else 1,
            **fam_gts,
            **fam_gts_lofs,
            **fam_gts_miss
        }
        
        rows.append(row)
        
    pheno_geno = pd.DataFrame(rows)

    gene_results = []
    
    for gene in gts:

        if len(pheno_geno[pheno_geno[gene] >= 1]['Total']) < 20:
            continue

        fam_gts = {gene: 1 if gts[gene][fam] >= 2 else 0 for gene in gts}
        fam_gts_lofs = {f'{gene}_lofs': 1 if gts_lofs[gene][fam] >= 2 else 0 for gene in gts_lofs}
        fam_gts_miss = {f'{gene}_miss': 1 if gts_miss[gene][fam] >= 2 else 0 for gene in gts_miss}

        fit = smf.ols(f'Rate ~ {gene} + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + Paternal_age + Maternal_age + Biobank + Source', data = pheno_geno).fit(cov_type = 'HC3')
        
        betas = fit.params[gene]
        pval_two_sided = fit.pvalues[gene]
        pval_one_sided = pval_two_sided / 2 if betas > 0 else 1 - (pval_two_sided / 2)
        
        gene_results.append({
            'gene': gene,
            'model': 'recessive' if recessive == True else 'additive',
            'phenotype': tp,
            'pval': pval_one_sided
        })

        # Predictions for carriers and non-carriers for top genes ----------
        
        if ((gene == 'REV1' and mutator == '' and tp == 'total') or
            (gene == 'LIG1' and mutator == '' and tp == 'CpG>TpG')):

            fam_pcs = get_pcs('X') # replace with random family
            fam_source = fams_source['X'] # replace with random family
            
            carrier_datapoint = pd.DataFrame({
                gene: [1],
                **{f'PC{i+1}': fam_pcs[i] for i in range(6)},
                'Paternal_age': [sum(fams_father_age.values())/len(fams_father_age)],
                'Maternal_age': [sum(fams_mother_age.values())/len(fams_mother_age)],
                'Biobank': [0 if 'ukb' in fam_source else 1],
                'Source': [0 if 'trio' in fam_source else 1]
            })

            predictions = fit.get_prediction(carrier_datapoint)
            predictions = predictions.summary_frame(alpha = 0.05)
            print(predictions['mean'].iloc[0], predictions['mean_ci_lower'].iloc[0], predictions['mean_ci_upper'].iloc[0])

            noncarrier_datapoint = pd.DataFrame({
                gene: [0],
                **{f'PC{i+1}': fam_pcs[i] for i in range(6)},
                'Paternal_age': [sum(fams_father_age.values())/len(fams_father_age)],
                'Maternal_age': [sum(fams_mother_age.values())/len(fams_mother_age)],
                'Biobank': [0 if 'ukb' in fam_source else 1],
                'Source': [0 if 'trio' in fam_source else 1]
            })
            
            predictions = fit.get_prediction(noncarrier_datapoint).summary_frame(alpha = 0.05)
            print(predictions['mean'].iloc[0], predictions['mean_ci_lower'].iloc[0], predictions['mean_ci_upper'].iloc[0])

        # ------------------------------------------------------------------

    gene_results_df = pd.DataFrame(gene_results)

    pvals_gc, lambda_gc = genomic_control(gene_results_df['pval'])
    print('Genomic control:', lambda_gc)
    gene_results_df['pval_gc'] = pvals_gc
    
    gene_results_df['qval'] = multipletests(gene_results_df['pval'], method = 'fdr_bh')[1]
    gene_results_df['qval_gc'] = multipletests(gene_results_df['pval_gc'], method = 'fdr_bh')[1]
    
    gene_results_df = pd.concat([gene_results_df, gene_results_df['gene'].apply(get_rates)], axis = 1)
    gene_results_df = gene_results_df.sort_values(by = ['qval', 'pval'])
    gene_results_df = gene_results_df[['gene', 'model', 'phenotype', 'N_carriers', 'N_noncarriers',
                                       'mean_rate_carriers', 'mean_rate_noncarriers', 'pval', 'qval', 
                                       'pval_gc', 'qval_gc']]

    if recessive == True:
        f = f'results/burden_test_recessive_{tp}.csv'

    else:
        f = f'results/burden_test_additive_{tp}.csv'
            
    gene_results_df.to_csv(f, index = False)

    plot_qq(gene_results_df['pval'], ci = True)

    print(gene_results_df[gene_results_df['pval'] <= 0.05][['gene', 'model', 'phenotype', 'N_carriers', 'N_noncarriers',
                                                            'mean_rate_carriers', 'mean_rate_noncarriers', 'pval', 'qval', 
                                                            'pval_gc', 'qval_gc']].to_string())

    if len(gene_results_df[gene_results_df['qval'] < 0.2]) > 0:

        # Comparison between LoF and missense effects ----------------------
    
        lofs_miss_results = []
    
        for gene in gene_results_df[gene_results_df['qval'] < 0.2]['gene']:
            
            fit = smf.ols(f'Rate ~ {gene}_lofs + {gene}_miss + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + Paternal_age + Maternal_age + Biobank + Source', data = pheno_geno).fit(cov_type = 'HC3')
            
            lofs_miss_results.append({
                'gene': gene,
                'model': 'recessive' if recessive == True else 'additive',
                'phenotype': tp,
                'N_LoF_carriers': len(pheno_geno[pheno_geno[f'{gene}_lofs'] >= 1]['Total']),
                'N_missense_carriers': len(pheno_geno[pheno_geno[f'{gene}_miss'] >= 1]['Total']),
                'LoF_pval': fit.pvalues.get(f'{gene}_lofs', np.nan),
                'missense_pval': fit.pvalues.get(f'{gene}_miss', np.nan),
                'f_test_pval': fit.f_test(f'{gene}_lofs = {gene}_miss').pvalue
            })
    
        lofs_miss_results_df = pd.DataFrame(lofs_miss_results)
        lofs_miss_results_df = lofs_miss_results_df[['gene', 'model', 'phenotype', 'N_LoF_carriers', 'N_missense_carriers',
                                                     'LoF_pval', 'missense_pval', 'f_test_pval']]

        if recessive == True:
            f = f'results/burden_test_lof_missense_comparison_recessive_{tp}.csv'
        else:
            f = f'results/burden_test_lof_missense_comparison_additive_{tp}.csv'
        
        lofs_miss_results_df.to_csv(f, index = False)
    
        print(lofs_miss_results_df[['gene', 'model', 'phenotype', 'N_LoF_carriers', 'N_missense_carriers',
                                    'LoF_pval', 'missense_pval', 'f_test_pval']].to_string())
    
        # ------------------------------------------------------------------
    
        # Significance of parental genotype sum x biobank ------------------
    
        genotype_biobank_results = []
    
        for gene in gene_results_df[gene_results_df['qval'] < 0.2]['gene']:
            
            fit = smf.ols(f'Rate ~ {gene} * Biobank + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + Paternal_age + Maternal_age + Source', data = pheno_geno).fit(cov_type = 'HC3')

            genotype_biobank_results.append({
                'gene': gene,
                'model': 'recessive' if recessive == True else 'additive',
                'phenotype': tp,
                'N_UKB_carriers': len(pheno_geno[(pheno_geno['Biobank'] == 0) & (pheno_geno[gene] >= 1)]['Total']),
                'N_AoU_carriers': len(pheno_geno[(pheno_geno['Biobank'] == 1) & (pheno_geno[gene] >= 1)]['Total']),
                'pval': fit.pvalues[f'{gene}:Biobank']
            })
    
        genotype_biobank_results_df = pd.DataFrame(genotype_biobank_results)
        genotype_biobank_results_df = genotype_biobank_results_df[['gene', 'model', 'phenotype', 'N_UKB_carriers', 'N_AoU_carriers', 'pval']]

        if recessive == True:
            f = f'results/burden_test_genotype_biobank_interaction_recessive_{tp}.csv'
        else:
            f = f'results/burden_test_genotype_biobank_interaction_additive_{tp}.csv'
        
        genotype_biobank_results_df.to_csv(f, index = False)
    
        print(genotype_biobank_results_df[['gene', 'model', 'phenotype', 'N_UKB_carriers', 'N_AoU_carriers', 'pval']].to_string())
    
        # ------------------------------------------------------------------

        # Significance of parental genotype sum x source -------------------
    
        genotype_source_results = []
    
        for gene in gene_results_df[gene_results_df['qval'] < 0.2]['gene']:
            
            fit = smf.ols(f'Rate ~ {gene} * Source + PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + Paternal_age + Maternal_age + Biobank', data = pheno_geno).fit(cov_type = 'HC3')
            
            genotype_source_results.append({
                'gene': gene,
                'model': 'recessive' if recessive == True else 'additive',
                'phenotype': tp,
                'N_trio_carriers': len(pheno_geno[(pheno_geno['Source'] == 0) & (pheno_geno[gene] >= 1)]['Total']),
                'N_sib_carriers': len(pheno_geno[(pheno_geno['Source'] == 1) & (pheno_geno[gene] >= 1)]['Total']),
                'pval': fit.pvalues[f'{gene}:Source']
            })
    
        genotype_source_results_df = pd.DataFrame(genotype_source_results)
        genotype_source_results_df = genotype_source_results_df[['gene', 'model', 'phenotype', 'N_trio_carriers', 'N_sib_carriers', 'pval']]

        if recessive == True:
            f = f'results/burden_test_genotype_source_interaction_recessive_{tp}.csv'
        else:
            f = f'results/burden_test_genotype_source_interaction_additive_{tp}.csv'
        
        genotype_source_results_df.to_csv(f, index = False)
    
        print(genotype_source_results_df[['gene', 'model', 'phenotype', 'N_trio_carriers', 'N_sib_carriers', 'pval']].to_string())
    
        # ------------------------------------------------------------------

    return gene_results_df['pval']

## Visual inspection

In [ ]:
def plot_spectrum(fams):

    colors = {
        'C>A': '#1f77b4',
        'C>G': '#000000',
        'C>T': '#e41a1c',
        'T>A': '#999999',
        'T>C': '#4daf4a',
        'T>G': '#ff7f00'
    }
    
    spectrum = merge_types(fams, fams_types)
    types = list(spectrum.keys())
    fracts = list(spectrum.values())
    groups = [colors[group[2:5]] for group in spectrum]

    plt.figure(figsize = (20, 5))
    plt.bar(types, fracts, color = groups, width = 0.8)
    plt.xticks(rotation = 90, fontsize = 7)

    plt.ylabel('Fraction')
    plt.xlabel('Mutation type')
    plt.show()
    plt.clf()

In [ ]:
def get_types_part(fam):
    
    types_part = {}

    for ref in ['C', 'T']:
        for alt in ['A', 'C', 'G', 'T']:
            if ref != alt:
                types_part[f'{ref}>{alt}'] = 0
                if ref == 'C' and alt == 'T':
                    types_part['CpG>TpG'] = 0
                        
    for tp in fams_types[fam]:
        tp_part = tp[2:5]
        if tp_part[0] == 'C' and tp_part[2] == 'T' and tp[-1] == 'G':
            tp_part = 'CpG>TpG'
        types_part[tp_part] += fams_types[fam][tp]
            
    return types_part

In [ ]:
def plot_spectrum_part(carriers, spectra = False):

    if len(carriers) == 0:
        return

    fams_types_part = {fam: get_types_part(fam) for fam in fams_types}

    noncarriers = [fam for fam in fams_types if fam not in carriers]

    print(len(carriers), len(noncarriers))
    
    carriers_types = merge_types(carriers, fams_types_part)
    noncarriers_types = merge_types(noncarriers, fams_types_part)
    
    types = list(carriers_types.keys())
    
    carriers_fracts = [cnt/sum(carriers_types.values()) 
                       for cnt in carriers_types.values()]
    noncarriers_fracts = [cnt/sum(noncarriers_types.values()) 
                          for cnt in noncarriers_types.values()]

    carriers_rates = [sum(fams_c_a_rates[fam] for fam in carriers)/len(carriers),
                      sum(fams_c_g_rates[fam] for fam in carriers)/len(carriers),
                      sum(fams_c_t_rates[fam] for fam in carriers)/len(carriers),
                      sum(fams_cpg_tpg_rates[fam] for fam in carriers)/len(carriers),
                      sum(fams_t_a_rates[fam] for fam in carriers)/len(carriers),
                      sum(fams_t_c_rates[fam] for fam in carriers)/len(carriers),
                      sum(fams_t_g_rates[fam] for fam in carriers)/len(carriers)]
    
    noncarriers_rates = [sum(fams_c_a_rates[fam] for fam in noncarriers)/len(noncarriers),
                         sum(fams_c_g_rates[fam] for fam in noncarriers)/len(noncarriers),
                         sum(fams_c_t_rates[fam] for fam in noncarriers)/len(noncarriers),
                         sum(fams_cpg_tpg_rates[fam] for fam in noncarriers)/len(noncarriers),
                         sum(fams_t_a_rates[fam] for fam in noncarriers)/len(noncarriers),
                         sum(fams_t_c_rates[fam] for fam in noncarriers)/len(noncarriers),
                         sum(fams_t_g_rates[fam] for fam in noncarriers)/len(noncarriers)]
    
    x = list(range(len(types)))

    if spectra == True:
        plt.bar(x, carriers_fracts, width = 0.4, color = 'red', label = 'Carriers')
        plt.bar([xi+0.4 for xi in x], noncarriers_fracts, width = 0.4, color = 'gray', label = 'Non-carriers')
        plt.xticks([xi+0.2 for xi in x], types)
        plt.ylabel('Fraction')
        plt.legend()
    else:
        dist = [carriers_rates[i]/noncarriers_rates[i]
                for i, _ in enumerate(carriers_rates)]
        cols = ['red' if d >= 1 else 'gray' for d in dist]
        plt.bar(x, [d - 1 for d in dist], bottom = 1, color = cols)
        plt.xticks(x, types)
        plt.ylabel('Fold change in mutation rate\n(carriers vs non-carriers)')
        
    plt.xlabel('Mutation type')
    plt.show()
    plt.clf()

In [ ]:
def list_types(fams):
    
    muts = []
    
    for fam in fams:
        for mut in fams_types[fam]:
            for _ in range(fams_types[fam][mut]):
                muts.append(mut)
            
    muts = pd.DataFrame({'Type': muts})
    
    return muts

In [ ]:
def plot_signatures(fams, beta = 0, plot = False):
    
    spectrum = list_types(fams)
    em = EM(spectrum, COSMIC(), beta = beta)
    em.infer()
    
    if plot:
        em.plot(first = 5)

    return em.fracts()

# Examples

## Burden test

In [ ]:
_ = gwas_rate_dosage('', 'total')

In [ ]:
_ = gwas_rate_dosage('', 'total', recessive = True)

In [ ]:
_ = gwas_rate_dosage('', 'CpG>TpG')

In [ ]:
_ = gwas_rate_dosage('', 'CpG>TpG', recessive = True)

In [ ]:
_ = gwas_rate_dosage('', 'C>G')

In [ ]:
_ = gwas_rate_dosage('', 'C>G', recessive = True)

In [ ]:
_ = gwas_rate_dosage('', 'C>T')

In [ ]:
_ = gwas_rate_dosage('', 'C>T', recessive = True)

In [ ]:
_ = gwas_rate_dosage('', 'T>C')

In [ ]:
_ = gwas_rate_dosage('', 'T>C', recessive = True)

In [ ]:
_ = gwas_rate_dosage('', 'C>A')

In [ ]:
_ = gwas_rate_dosage('', 'C>A', recessive = True)

## Merge result files

In [ ]:
models = ['additive', 'recessive']
tps = ['total', 'CpG>TpG', 'C>G', 'C>T', 'T>C', 'C>A']
df = pd.concat([pd.read_csv(f'results/burden_test_{model}_{tp}.csv') for model in models for tp in tps], ignore_index = True)
df.to_csv('results/burden_test.csv', index = False)

In [ ]:
models = ['additive', 'recessive']
tps = ['total', 'CpG>TpG', 'C>G', 'C>T', 'T>C', 'C>A']
df = pd.concat([pd.read_csv(f'results/burden_test_lof_missense_comparison_{model}_{tp}.csv') for model in models for tp in tps
               if os.path.exists(f'results/burden_test_lof_missense_comparison_{model}_{tp}.csv')], ignore_index = True)
df.to_csv('results/burden_test_lof_missense_comparison.csv', index = False)

In [ ]:
models = ['additive', 'recessive']
tps = ['total', 'CpG>TpG', 'C>G', 'C>T', 'T>C', 'C>A']
df = pd.concat([pd.read_csv(f'results/burden_test_genotype_biobank_interaction_{model}_{tp}.csv') for model in models for tp in tps
               if os.path.exists(f'results/burden_test_genotype_biobank_interaction_{model}_{tp}.csv')], ignore_index = True)
df.to_csv('results/burden_test_genotype_biobank_interaction.csv', index = False)

### NNLS

In [ ]:
carriers = get_carriers(1, '')

for gene in ['REV1', 'LIG1']:

    y_carriers = merge_types(list(carriers[gene]), fams_types)
    y_carriers = np.array([y_carriers.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
    betas_carriers, _ = nnls(cosmic_X, y_carriers)
    y_reconstructed = cosmic_X @ betas_carriers
    cos_sim_carriers = cosine_similarity(y_carriers.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0]
    
    noncarriers = [fam for fam in fams_types if fam not in carriers[gene]]
    y_noncarriers = merge_types(noncarriers, fams_types)
    y_noncarriers = np.array([y_noncarriers.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
    betas_noncarriers, _ = nnls(cosmic_X, y_noncarriers)
    y_reconstructed = cosmic_X @ betas_noncarriers
    cos_sim_noncarriers = cosine_similarity(y_noncarriers.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0]
    
    print(gene, cos_sim_carriers, cos_sim_noncarriers)

    cos_sim_noncarriers_subsample = []
    for _ in range(100):
        noncarriers_subsample = np.random.choice(noncarriers, size = len(carriers[gene]), replace = False)
        y = merge_types(noncarriers_subsample, fams_types)
        y = np.array([y.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
        betas_noncarriers_subsample, _ = nnls(cosmic_X, y)
        y_reconstructed = cosmic_X @ betas_noncarriers_subsample
        cos_sim_noncarriers_subsample.append(cosine_similarity(y.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0])
    print(sum(cos_sim_noncarriers_subsample)/len(cos_sim_noncarriers_subsample))
    
    with open(f'results/NNLS_{gene}.csv', 'w') as f:
        f.write('gene,carriers,noncarriers,pval\n')
        for i, beta_carriers in enumerate(betas_carriers):
            if beta_carriers/sum(betas_carriers) == 0 and betas_noncarriers[i]/sum(betas_noncarriers) == 0:
                f.write(f"{cosmic_df.columns[i]},{round(beta_carriers/sum(betas_carriers), 4)},{round(betas_noncarriers[i]/sum(betas_noncarriers), 4)},\n")
            else:
                change_df = sbs_permutation_carrier_status(1, '', [gene], [cosmic_df.columns[i]], '')
                f.write(f"{cosmic_df.columns[i]},{round(beta_carriers/sum(betas_carriers), 4)},{round(betas_noncarriers[i]/sum(betas_noncarriers), 4)},{round(change_df.iloc[0]['pval'],4)}\n")

In [ ]:
carriers = get_carriers(1, '')

for gene in ['REV1', 'LIG1']:

    y_carriers = merge_types(list(carriers[gene]), fams_types)
    y_carriers = np.array([y_carriers.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
    betas_carriers, _ = nnls(cosmic_X, y_carriers)
    y_reconstructed = cosmic_X @ betas_carriers
    cos_sim_carriers = cosine_similarity(y_carriers.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0]
    
    noncarriers = [fam for fam in fams_types if fam not in carriers[gene]]
    y_noncarriers = merge_types(noncarriers, fams_types)
    y_noncarriers = np.array([y_noncarriers.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
    betas_noncarriers, _ = nnls(cosmic_X, y_noncarriers)
    y_reconstructed = cosmic_X @ betas_noncarriers
    cos_sim_noncarriers = cosine_similarity(y_noncarriers.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0]
    
    print(gene, cos_sim_carriers, cos_sim_noncarriers)

    cos_sim_noncarriers_subsample = []
    for _ in range(100):
        noncarriers_subsample = np.random.choice(noncarriers, size = len(carriers[gene]), replace = False)
        y = merge_types(noncarriers_subsample, fams_types)
        y = np.array([y.get(mut, 0) for mut in cosmic_mutation_types], dtype = float)
        betas_noncarriers_subsample, _ = nnls(cosmic_X, y)
        y_reconstructed = cosmic_X @ betas_noncarriers_subsample
        cos_sim_noncarriers_subsample.append(cosine_similarity(y.reshape(1, -1), y_reconstructed.reshape(1, -1))[0, 0])
    print(sum(cos_sim_noncarriers_subsample)/len(cos_sim_noncarriers_subsample))
    
    with open(f'results/NNLS_{gene}.csv', 'w') as f:
        f.write('gene,carriers,noncarriers,pval\n')
        for i, beta_carriers in enumerate(betas_carriers):
            if beta_carriers/sum(betas_carriers) == 0 and betas_noncarriers[i]/sum(betas_noncarriers) == 0:
                f.write(f"{cosmic_df.columns[i]},{round(beta_carriers/sum(betas_carriers), 4)},{round(betas_noncarriers[i]/sum(betas_noncarriers), 4)},\n")
            else:
                change_df = sbs_permutation_carrier_status(1, '', [gene], [cosmic_df.columns[i]], '')
                f.write(f"{cosmic_df.columns[i]},{round(beta_carriers/sum(betas_carriers), 4)},{round(betas_noncarriers[i]/sum(betas_noncarriers), 4)},{round(change_df.iloc[0]['pval'],4)}\n")

### E-M

In [ ]:
carriers = get_carriers(1, '')
noncarriers = [fam for fam in fams_types if fam not in carriers['REV1']]

carriers_cnt = sum(sum(fams_types[fam].values()) for fam in carriers['REV1'])
noncarriers_cnt = sum(sum(fams_types[fam].values()) for fam in noncarriers)

beta = 0

sigs_carriers = plot_signatures(carriers['REV1'], beta = beta*carriers_cnt)
sigs_noncarriers = plot_signatures(noncarriers, beta = beta*noncarriers_cnt)

with open(f'results/EM_REV1_beta{beta}.csv', 'w') as f:
    f.write('gene,carriers,noncarriers\n')
    for i, (sbs_carriers, fract_carriers) in enumerate(sigs_carriers):
        if fract_carriers == 0 and sigs_noncarriers[i][1] == 0:
            continue
        f.write(f'{sbs_carriers},{round(fract_carriers, 4)},{round(sigs_noncarriers[i][1], 4)}\n')

In [ ]:
carriers = get_carriers(1, '')
noncarriers = [fam for fam in fams_types if fam not in carriers['LIG1']]

carriers_cnt = sum(sum(fams_types[fam].values()) for fam in carriers['LIG1'])
noncarriers_cnt = sum(sum(fams_types[fam].values()) for fam in noncarriers)

beta = 0

sigs_carriers = plot_signatures(carriers['LIG1'], beta = beta*carriers_cnt)
sigs_noncarriers = plot_signatures(noncarriers, beta = beta*noncarriers_cnt)

with open(f'results/EM_LIG1_beta{beta}.csv', 'w') as f:
    f.write('gene,carriers,noncarriers\n')
    for i, (sbs_carriers, fract_carriers) in enumerate(sigs_carriers):
        if fract_carriers == 0 and sigs_noncarriers[i][1] == 0:
            continue
        f.write(f'{sbs_carriers},{round(fract_carriers, 4)},{round(sigs_noncarriers[i][1], 4)}\n')